In [ ]:
#IMPORT LIBRARIES AND DATASET

import pandas as pd # Import the pandas library for data manipulation, aliasing it as 'pd'
import numpy as np # Import the numpy library for numerical operations, aliasing it as 'np'
import matplotlib.pyplot as plt # Import the matplotlib.pyplot library for plotting, aliasing it as 'plt'
from math import sqrt # Import the 'sqrt' function from the 'math' module

#Importing Dataset

ratings_df = pd.read_csv('ratings.csv') # Load the 'ratings.csv' file into a pandas DataFrame called 'ratings_df'
ratings_df.head() # Display the first few rows of the 'ratings_df' DataFrame

#Importing the movies dataset from where we have got the ratings

movies_df = pd.read_csv('movies.csv') # Load the 'movies.csv' file into a pandas DataFrame called 'movies_df'
movies_df.head() # Display the first few rows of the 'movies_df' DataFrame

#DATA PREPROCESSING

"""In movies dataset we have year along with the title"""
"""So first we will extract year from title and assign it to a new column"""

# Extract the year (e.g., '(YYYY)') from the 'title' column using a regular expression
movies_df['year'] = movies_df.title.str.extract('(\(\d\d\d\d\))', expand = True)
movies_df.head() # Display the first few rows to see the new 'year' column

#Remove parantheses from year

# Extract only the four digits (YYYY) from the 'year' column, removing parentheses
movies_df['year'] = movies_df.year.str.extract('(\d\d\d\d)', expand=True)
movies_df.head() # Display the first few rows to see the updated 'year' column

#Remove year from title

# Remove the year (e.g., '(YYYY)') from the 'title' column using a regular expression
movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '', regex=True)
movies_df.head() # Display the first few rows to see the updated 'title' column

#Remove all the whitespaces from title

# Apply a string strip function used to create function lambda to remove leading/trailing whitespaces from each title
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())

#Convert Genres into a list

# Split the 'genres' string by the '|' character into a list of genres
movies_df['genres'] = movies_df.genres.str.split('|')
movies_df.head() # Display the first few rows to see the 'genres' column as lists

#One Hot Encoding of Genres

movies_copy = movies_df.copy() # Create a copy of the movies DataFrame to perform one-hot encoding

# Iterate through each row of the movies_df
for index, row in movies_df.iterrows():
  # For each genre in the 'genres' list of the current row
  for genre in row['genres']:
    # Set the value to 1 in the 'movies_copy' DataFrame at the corresponding index and genre column
    movies_copy.at[index, genre] = 1

movies_copy.head() # Display the first few rows of the one-hot encoded DataFrame

#Filling NAN values with 0

movies_copy = movies_copy.fillna(0) # Replace all NaN (Not a Number) values with 0
movies_copy.head() # Display the first few rows after filling NaNs

#Now let's check ratings dataset

ratings_df.head() # Display the first few rows of the 'ratings_df' DataFrame

#Timestamp column is not necessary, so we can drop it

# Drop the 'timestamp' column from the 'ratings_df' DataFrame
ratings_df = ratings_df.drop(['timestamp'], axis=1)
ratings_df.head() # Display the first few rows after dropping the 'timestamp' column

#CONTENT BASED RECOMMENDATION SYSTEM

"""Let's start by taking User Input for ratings of different movies"""

# Define a list of dictionaries representing user's movie ratings
user_input = [
              {'title' : 'Grand Slam', 'rating' : 5.6},
              {'title' : 'Zero', 'rating' : 7},# dict{ key:value}
              {'title' : 'Jumanji', 'rating' : 8.5},
              {'title' : 'Toy Story', 'rating' : 4.5}
]

movies_input = pd.DataFrame(user_input) # Convert the user input list into a pandas DataFrame
movies_input # Display the 'movies_input' DataFrame

#Add movieID to user input
#First we will filter selected movies from original dataset

# Filter 'movies_df' to get movie information for the titles provided in 'user_input'
input_id = movies_df[movies_df['title'].isin(movies_input['title'].tolist())]

#Merging the two datasets

# Merge 'input_id' with 'movies_input' based on the 'title' column to add 'movieId', 'genres', and 'year'
movies_input = pd.merge(input_id, movies_input)
movies_input # Display the merged 'movies_input' DataFrame

#Drop the unnecessary columns like genres and year

# Drop the 'genres' and 'year' columns as they are no longer needed for this step
movies_input = movies_input.drop(['genres','year'], axis=1)
movies_input # Display the 'movies_input' DataFrame after dropping columns

#Now we will check for same movies given in input in original dataset

# Filter the one-hot encoded 'movies_copy' DataFrame to include only the movies rated by the user
movies_user = movies_copy[movies_copy['movieId'].isin(movies_input['movieId'].tolist())]
movies_user # Display the 'movies_user' DataFrame

#Reset index of this dataset

# Reset the index of 'movies_user' and drop the old index
movies_user = movies_user.reset_index(drop=True)
movies_user # Display the 'movies_user' DataFrame with reset index

#Create a Genre Table out of this dataset

# Create a 'UserGenreTable' by dropping 'movieId', 'title', 'genres', and 'year' from 'movies_user'
# This table now represents the genres of movies the user rated, with 1s for present genres
UserGenreTable = movies_user.drop(['movieId','title','genres','year'], axis=1)
UserGenreTable # Display the 'UserGenreTable'

#dot product to get weights

# Calculate the 'UserProfile' by taking the dot product of the transposed 'UserGenreTable'
# and the 'rating' column from 'movies_input'. This gives a weighted sum of genres based on ratings.
UserProfile = UserGenreTable.transpose().dot(movies_input['rating'])

#User Profile for every genre

UserProfile # Display the 'UserProfile', which shows the preference score for each genre

#Create a genre table for every movie in original datset

# Set the 'movieId' as the index for 'movies_copy' to facilitate genre-based operations
GenreTable = movies_copy.set_index(movies_copy['movieId'])
GenreTable # Display the 'GenreTable'

# Drop 'movieId', 'title', 'genres', and 'year' from 'GenreTable', leaving only genre columns
GenreTable = GenreTable.drop(['movieId','title','genres','year'], axis=1)
GenreTable.head() # Display the head of the processed 'GenreTable'

#Final Recommendation value for each movie

# Calculate recommendation scores for each movie by multiplying its genres by the 'UserProfile' weights,
# summing them up, and then normalizing by the sum of 'UserProfile' weights.
Recommendation_df = ((GenreTable*UserProfile).sum(axis=1))/UserProfile.sum()
Recommendation_df.head() # Display the head of the 'Recommendation_df'

#Sort the values to get movies with high recommendation values

# Sort the recommendation scores in descending order to find the highest recommended movies
Recommendation_df = Recommendation_df.sort_values(ascending=False)
Recommendation_df.head() # Display the head of the sorted 'Recommendation_df'

#Final recommendation table for movies

# Retrieve the top 20 recommended movies by filtering 'movies_df' using the movieIds
# from the top 20 keys (movieIds) of 'Recommendation_df'
RecommendationTable =  movies_df.loc[movies_df['movieId'].isin(Recommendation_df.head(20).keys())]
RecommendationTable # Display the final 'RecommendationTable' of recommended movies